## P000249DS Conversational AI Business Intelligence Dashboard for SMEs

### Project Overview

This notebook implements a complete ETL pipeline and analytics engine for an SME business intelligence system. The solution processes financial, sales, marketing, and customer data to answer critical business questions.

### Step 1: Setup and Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("Libraries imported successfully!")

In [4]:
!pip install psycopg2-binary

     |████████████████████████████████| 385 kB 10.3 MB/s eta 0:00:01
    ERROR: Command errored out with exit status 1:
     command: /Users/saumyabandara/opt/anaconda3/bin/python -c 'import sys, setuptools, tokenize; sys.argv[0] = '"'"'/private/var/folders/xq/rtcrgcnd6l73tk4qh3f2srtr0000gn/T/pip-install-m7lwwsl4/psycopg2-binary_8d198a7ca5c6452986993aea8b7a9d6e/setup.py'"'"'; __file__='"'"'/private/var/folders/xq/rtcrgcnd6l73tk4qh3f2srtr0000gn/T/pip-install-m7lwwsl4/psycopg2-binary_8d198a7ca5c6452986993aea8b7a9d6e/setup.py'"'"';f=getattr(tokenize, '"'"'open'"'"', open)(__file__);code=f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' egg_info --egg-base /private/var/folders/xq/rtcrgcnd6l73tk4qh3f2srtr0000gn/T/pip-pip-egg-info-r2nef1qi
         cwd: /private/var/folders/xq/rtcrgcnd6l73tk4qh3f2srtr0000gn/T/pip-install-m7lwwsl4/psycopg2-binary_8d198a7ca5c6452986993aea8b7a9d6e/
    Complete output (23 lines):
    running egg_info
    cre

### Step 2: Load and Explore Data

In [7]:
# Define file paths
import os

# Load all CSV files
campaigns_df = pd.read_csv('conversational_bi_sample_database/campaigns.csv')
cash_balances_df = pd.read_csv('conversational_bi_sample_database/cash_balances.csv')
companies_df = pd.read_csv('conversational_bi_sample_database/companies.csv')
customer_metrics_df = pd.read_csv('conversational_bi_sample_database/customer_metrics.csv')
customers_df = pd.read_csv('conversational_bi_sample_database/customers.csv')
data_sources_df = pd.read_csv('conversational_bi_sample_database/data_sources.csv')
expenses_df = pd.read_csv('conversational_bi_sample_database/expenses.csv')
marketing_performance_df = pd.read_csv('conversational_bi_sample_database/marketing_performance.csv')
order_items_df = pd.read_csv('conversational_bi_sample_database/order_items.csv')
orders_df = pd.read_csv('conversational_bi_sample_database/orders.csv')
products_df = pd.read_csv('conversational_bi_sample_database/products.csv')
transactions_df = pd.read_csv('conversational_bi_sample_database/transactions.csv')
users_df = pd.read_csv('conversational_bi_sample_database/users.csv')

print("All CSV files loaded successfully!")
print(f"\nFiles loaded:")
print(f"- Campaigns: {len(campaigns_df)} records")
print(f"- Cash Balances: {len(cash_balances_df)} records")
print(f"- Companies: {len(companies_df)} records")
print(f"- Customer Metrics: {len(customer_metrics_df)} records")
print(f"- Customers: {len(customers_df)} records")
print(f"- Expenses: {len(expenses_df)} records")
print(f"- Marketing Performance: {len(marketing_performance_df)} records")
print(f"- Order Items: {len(order_items_df)} records")
print(f"- Orders: {len(orders_df)} records")
print(f"- Products: {len(products_df)} records")
print(f"- Transactions: {len(transactions_df)} records")
print(f"- Users: {len(users_df)} records")

All CSV files loaded successfully!

Files loaded:
- Campaigns: 4 records
- Cash Balances: 182 records
- Companies: 1 records
- Customer Metrics: 120 records
- Customers: 120 records
- Expenses: 439 records
- Marketing Performance: 683 records
- Order Items: 2387 records
- Orders: 1420 records
- Products: 6 records
- Transactions: 621 records
- Users: 3 records


### Step 3: Data Cleaning and Preprocessing

In [8]:
# Function to display data quality report
def data_quality_report(df, name):
    """Generate data quality report for a dataframe"""
    print(f"\n{'='*60}")
    print(f"Data Quality Report: {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nMissing Values:")
    print(df.isnull().sum())
    print(f"\nData Types:")
    print(df.dtypes)
    print(f"\nDuplicate Rows: {df.duplicated().sum()}")
    
    # Display basic statistics for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\nBasic Statistics for Numeric Columns:")
        print(df[numeric_cols].describe())

# Run quality reports
data_quality_report(campaigns_df, "Campaigns")
data_quality_report(cash_balances_df, "Cash Balances")
data_quality_report(customers_df, "Customers")
data_quality_report(orders_df, "Orders")
data_quality_report(transactions_df, "Transactions")


Data Quality Report: Campaigns
Shape: (4, 7)

Missing Values:
campaign_id      0
company_id       0
campaign_name    0
platform         0
start_date       0
end_date         0
budget           0
dtype: int64

Data Types:
campaign_id       int64
company_id        int64
campaign_name    object
platform         object
start_date       object
end_date         object
budget            int64
dtype: object

Duplicate Rows: 0

Basic Statistics for Numeric Columns:
       campaign_id  company_id        budget
count     4.000000         4.0      4.000000
mean      2.500000         1.0  10375.000000
std       1.290994         0.0   6446.898479
min       1.000000         1.0   2500.000000
25%       1.750000         1.0   7375.000000
50%       2.500000         1.0  10500.000000
75%       3.250000         1.0  13500.000000
max       4.000000         1.0  18000.000000

Data Quality Report: Cash Balances
Shape: (182, 5)

Missing Values:
balance_id         0
company_id         0
date               0
o

In [9]:
# Convert date columns to datetime format
date_columns = {
    'campaigns_df': ['start_date', 'end_date'],
    'cash_balances_df': ['date'],
    'companies_df': ['created_at'],
    'customer_metrics_df': ['last_order_date'],
    'customers_df': ['created_at'],
    'data_sources_df': ['last_sync_at'],
    'expenses_df': ['date'],
    'marketing_performance_df': ['date'],
    'orders_df': ['order_date'],
    'transactions_df': ['date'],
    'users_df': ['created_at']
}

# Convert dates
for df_name, cols in date_columns.items():
    df = eval(df_name)
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    print(f"Converted dates for {df_name}")

print("\nDate conversion complete!")

Converted dates for campaigns_df
Converted dates for cash_balances_df
Converted dates for companies_df
Converted dates for customer_metrics_df
Converted dates for customers_df
Converted dates for data_sources_df
Converted dates for expenses_df
Converted dates for marketing_performance_df
Converted dates for orders_df
Converted dates for transactions_df
Converted dates for users_df

Date conversion complete!


In [10]:
# Clean and prepare orders data
# Filter only completed orders for accurate analysis
orders_df = orders_df[orders_df['status'] == 'completed'].copy()
print(f"Completed orders: {len(orders_df)}")

# Add profit calculation to order_items
order_items_df['profit'] = (order_items_df['unit_price'] - order_items_df['cost_price']) * order_items_df['quantity']

# Calculate order-level metrics
order_profit = order_items_df.groupby('order_id')['profit'].sum().reset_index()
order_profit.columns = ['order_id', 'total_profit']

# Merge profit with orders
orders_df = orders_df.merge(order_profit, on='order_id', how='left')
orders_df['total_profit'] = orders_df['total_profit'].fillna(0)

print(f"Orders with profit calculated: {len(orders_df)}")

Completed orders: 1390
Orders with profit calculated: 1390


### Step 4: Design and Implement Star Schema for PostgreSQL

In [11]:
# PostgreSQL Connection Configuration
# Update these values based on your PostgreSQL setup
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'sme_bi_dashboard',
    'user': 'postgres',
    'password': 'your_password'  # Replace with actual password
}

# Create connection string
def get_connection_string():
    return f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

In [12]:
# Define Star Schema Tables

# Dimension Tables
dim_tables = {
    'dim_company': companies_df,
    'dim_customer': customers_df,
    'dim_product': products_df,
    'dim_campaign': campaigns_df,
    'dim_date': None,  # Will create date dimension
    'dim_user': users_df
}

# Fact Tables
fact_tables = {
    'fact_orders': orders_df,
    'fact_order_items': order_items_df,
    'fact_transactions': transactions_df,
    'fact_expenses': expenses_df,
    'fact_marketing_performance': marketing_performance_df,
    'fact_cash_balances': cash_balances_df,
    'fact_customer_metrics': customer_metrics_df
}

print("Star Schema structure defined:")
print("\nDimension Tables:", list(dim_tables.keys()))
print("Fact Tables:", list(fact_tables.keys()))

Star Schema structure defined:

Dimension Tables: ['dim_company', 'dim_customer', 'dim_product', 'dim_campaign', 'dim_date', 'dim_user']
Fact Tables: ['fact_orders', 'fact_order_items', 'fact_transactions', 'fact_expenses', 'fact_marketing_performance', 'fact_cash_balances', 'fact_customer_metrics']


In [13]:
# Create Date Dimension Table
def create_date_dimension(start_date='2024-01-01', end_date='2026-12-31'):
    """Create a comprehensive date dimension table"""
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    
    date_dim = pd.DataFrame({
        'date_id': date_range.strftime('%Y%m%d').astype(int),
        'date': date_range,
        'year': date_range.year,
        'quarter': date_range.quarter,
        'month': date_range.month,
        'month_name': date_range.strftime('%B'),
        'week': date_range.isocalendar().week,
        'day': date_range.day,
        'day_name': date_range.strftime('%A'),
        'day_of_week': date_range.dayofweek,
        'is_weekend': (date_range.dayofweek >= 5).astype(int),
        'is_month_start': date_range.is_month_start.astype(int),
        'is_month_end': date_range.is_month_end.astype(int),
        'year_month': date_range.strftime('%Y-%m'),
    })
    
    return date_dim

date_dim_df = create_date_dimension()
print(f"Date dimension created with {len(date_dim_df)} rows")
print("\nSample:")
print(date_dim_df.head())

Date dimension created with 1096 rows

Sample:
             date_id       date  year  quarter  month month_name  week  day  \
2024-01-01  20240101 2024-01-01  2024        1      1    January     1    1   
2024-01-02  20240102 2024-01-02  2024        1      1    January     1    2   
2024-01-03  20240103 2024-01-03  2024        1      1    January     1    3   
2024-01-04  20240104 2024-01-04  2024        1      1    January     1    4   
2024-01-05  20240105 2024-01-05  2024        1      1    January     1    5   

             day_name  day_of_week  is_weekend  is_month_start  is_month_end  \
2024-01-01     Monday            0           0               1             0   
2024-01-02    Tuesday            1           0               0             0   
2024-01-03  Wednesday            2           0               0             0   
2024-01-04   Thursday            3           0               0             0   
2024-01-05     Friday            4           0               0             0  

In [14]:
# Prepare data for PostgreSQL loading

def prepare_for_database(df, table_name):
    """Prepare dataframe for database insertion"""
    df_clean = df.copy()
    
    # Replace NaN with None for PostgreSQL
    df_clean = df_clean.where(pd.notnull(df_clean), None)
    
    # Convert datetime to string for PostgreSQL
    for col in df_clean.columns:
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].astype(str)
    
    print(f"Prepared {table_name}: {len(df_clean)} rows")
    return df_clean

# Prepare all tables
prepared_tables = {}

for table_name, df in dim_tables.items():
    if df is not None:
        prepared_tables[table_name] = prepare_for_database(df, table_name)

for table_name, df in fact_tables.items():
    prepared_tables[table_name] = prepare_for_database(df, table_name)

prepared_tables['dim_date'] = prepare_for_database(date_dim_df, 'dim_date')

print(f"\nTotal tables prepared: {len(prepared_tables)}")

Prepared dim_company: 1 rows
Prepared dim_customer: 120 rows
Prepared dim_product: 6 rows
Prepared dim_campaign: 4 rows
Prepared dim_user: 3 rows
Prepared fact_orders: 1390 rows
Prepared fact_order_items: 2387 rows
Prepared fact_transactions: 621 rows
Prepared fact_expenses: 439 rows
Prepared fact_marketing_performance: 683 rows
Prepared fact_cash_balances: 182 rows
Prepared fact_customer_metrics: 120 rows
Prepared dim_date: 1096 rows

Total tables prepared: 13
